# chl-gapfill on a free GPU (Colab / Kaggle)

The same CLI and code as on CPU, on synthetic data by default: this notebook proves the GPU path
and records the timing; the published results need the real data (Option A).
Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!git clone https://github.com/Wina-w/chlorophyll-convlstm-gapfilling.git repo
%cd repo
!pip -q install -e ".[track]"
import torch; print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

In [ ]:
# Option A: the real data from NOAA ERDDAP (target and sensors); then skip the CHL_DATA_DIR line
# and use -c configs/convlstm.yaml (without --set smoke=false) in the cells below
# !chl-gapfill download -c configs/convlstm.yaml
# Option B: synthetic data (always works)
!python scripts/make_synthetic_data.py --out data_synth
%env CHL_DATA_DIR=/content/repo/data_synth
%env CHL_MODEL_DIR=/content/repo/models
%env CHL_OUTPUT_DIR=/content/repo/outputs_smoke

In [ ]:
import time
t0 = time.time()
!chl-gapfill train -c configs/synthetic_smoke.yaml --set smoke=false training.n_epochs=30 --no-mlflow
print(f'wall time: {time.time()-t0:.0f} s')  # record CPU vs GPU in the README

In [ ]:
!chl-gapfill evaluate -c configs/synthetic_smoke.yaml --set smoke=false --n-boot 200
!chl-gapfill predict -c configs/synthetic_smoke.yaml --set smoke=false --date 2022-04-20 --png
from IPython.display import Image, display
display(Image('outputs_smoke/filled_2022-04-20.png'))

In [ ]:
# keep the artifacts
from google.colab import drive  # noqa
drive.mount('/content/drive')
!cp -r models outputs_smoke /content/drive/MyDrive/chl_gapfill_run/